# Olist Brazilian E-Commerce Dataset

## Dataset Understanding

**HVIA Data & AI Solutions Task**  
**Prepared by:** Youssef Atef Tayh  
**Dataset:** Brazilian E-Commerce Public Dataset by Olist  
**Period:** 2016–2018  
**Stage:** Dataset Understanding

This notebook establishes the structure of the dataset before data quality checks and exploratory analysis.

## Learning Note

The goal is to understand the data before trying to explain the business.

For each stage, the notebook follows:

**Markdown → Code → Output → Interpretation**

No records are modified in this notebook.

## Roadmap

1. Project Objective
2. Dataset Context
3. Project Setup
4. Dataset Files
5. Import Libraries
6. Load Dataset
7. Table Inventory
8. Data Grain
9. First Look at the Data
10. Column Overview
11. Data Types
12. Data Dictionary
13. Table Relationships
14. Primary Key and Relationship Checks
15. Missing Values
16. Duplicate Records
17. Order Status
18. Order Date Range
19. Order Lifecycle
20. Geographic Structure
21. Category Translation Structure
22. Initial Structural Observations
23. What We Learned
24. Dataset Understanding Summary
25. Next Stage: Data Quality

# 1. Project Objective

Understand the Olist dataset structure before starting data quality analysis and business-focused exploratory analysis.

This stage covers:

- Tables and their sizes
- Data grain
- Important columns
- Data types
- Relationships
- Missing values
- Duplicate records
- Order lifecycle
- Geographic structure

# 2. Dataset Context

The Brazilian E-Commerce Public Dataset by Olist contains information about customers, orders, order items, payments, reviews, products, sellers, geolocation, and product-category translation.

The order purchase period represented in the dataset is **2016–2018**.

This notebook focuses on the historical dataset structure. Current Olist operations should not be assumed to be identical to the period represented by the dataset.

# 3. Project Setup

In [1]:
# Data analysis
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# File paths and display
from pathlib import Path
from IPython.display import display

# Hide unnecessary warnings
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

In [2]:
current_folder = Path.cwd()

if (current_folder / "data" / "raw").exists():
    PROJECT_ROOT = current_folder
elif (current_folder.parent / "data" / "raw").exists():
    PROJECT_ROOT = current_folder.parent
else:
    raise FileNotFoundError("Could not find the project data/raw folder.")

DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data folder: {DATA_DIR}")

Project root: E:\Interniship\HVIA - Data & AI Intern 2026\Tasks\Daily Tasks\Task 1\Olist_HVIA_Youssef_Atef_Tayh
Raw data folder: E:\Interniship\HVIA - Data & AI Intern 2026\Tasks\Daily Tasks\Task 1\Olist_HVIA_Youssef_Atef_Tayh\data\raw


# 4. Dataset Files

Before loading the data, verify that all expected CSV files are present.

In [3]:
expected_files = [
    "olist_customers_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "product_category_name_translation.csv",
]

file_check = pd.DataFrame([
    {
        "File": file_name,
        "Status": "Available" if (DATA_DIR / file_name).exists() else "Missing",
        "Path": str(DATA_DIR / file_name),
    }
    for file_name in expected_files
])

display(file_check)

,File,Status,Path
0,olist_customers_dataset.csv,Available,E:\Interniship\HVIA - Data & AI Intern 2026\Ta...
1,olist_orders_dataset.csv,Available,E:\Interniship\HVIA - Data & AI Intern 2026\Ta...
2,olist_order_items_dataset.csv,Available,E:\Interniship\HVIA - Data & AI Intern 2026\Ta...
3,olist_order_payments_dataset.csv,Available,E:\Interniship\HVIA - Data & AI Intern 2026\Ta...
4,olist_order_reviews_dataset.csv,Available,E:\Interniship\HVIA - Data & AI Intern 2026\Ta...
5,olist_products_dataset.csv,Available,E:\Interniship\HVIA - Data & AI Intern 2026\Ta...
6,olist_sellers_dataset.csv,Available,E:\Interniship\HVIA - Data & AI Intern 2026\Ta...
7,olist_geolocation_dataset.csv,Available,E:\Interniship\HVIA - Data & AI Intern 2026\Ta...
8,product_category_name_translation.csv,Available,E:\Interniship\HVIA - Data & AI Intern 2026\Ta...


# 5. Import Libraries

All libraries used in the project are imported in one place.

# 6. Load Dataset

In [4]:
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
order_items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv")
geolocation = pd.read_csv(DATA_DIR / "olist_geolocation_dataset.csv")
translation = pd.read_csv(DATA_DIR / "product_category_name_translation.csv")

tables = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "translation": translation,
}

print(f"Loaded {len(tables)} tables successfully.")

Loaded 9 tables successfully.


### Load Interpretation

The nine source tables are now available as separate DataFrames. The raw CSV files are kept unchanged.

# 7. Table Inventory

In [5]:
table_inventory = pd.DataFrame([
    {"Table": name, "Rows": len(table), "Columns": len(table.columns)}
    for name, table in tables.items()
])
display(table_inventory)

,Table,Rows,Columns
0,customers,99441,5
1,orders,99441,8
2,order_items,112650,7
3,payments,103886,5
4,reviews,99224,7
5,products,32951,9
6,sellers,3095,4
7,geolocation,1000163,5
8,translation,71,2


### Table Inventory Interpretation

The tables have different sizes because they represent different business entities and different data grains.

# 8. Data Grain

In [6]:
data_grain = pd.DataFrame([
    ["Customers", "One customer record"],
    ["Orders", "One order"],
    ["Order Items", "One item within an order"],
    ["Payments", "One payment record"],
    ["Reviews", "One review record"],
    ["Products", "One product"],
    ["Sellers", "One seller"],
    ["Geolocation", "One geographic record associated with a postal-code prefix"],
    ["Translation", "One product-category translation"],
], columns=["Table", "One Row Represents"])
display(data_grain)

,Table,One Row Represents
0,Customers,One customer record
1,Orders,One order
2,Order Items,One item within an order
3,Payments,One payment record
4,Reviews,One review record
5,Products,One product
6,Sellers,One seller
7,Geolocation,One geographic record associated with a postal...
8,Translation,One product-category translation


### Data Grain Interpretation

Data grain is important because repeated identifiers can be valid. For example, one order can have multiple item rows and multiple payment records.

# 9. First Look at the Data

### Customers

Customer records include customer identifiers and location fields.

In [7]:
display(customers.head())

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


### Orders

Order records include status and lifecycle timestamps.

In [8]:
display(orders.head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


### Order Items

Order items connect orders with products and sellers and contain item-level price fields.

In [9]:
display(order_items.head())

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


### Payments

Payment records contain payment type, installments, and value.

In [10]:
display(payments.head())

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


### Reviews

Review records contain scores, comments, and review dates.

In [11]:
display(reviews.head())

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


### Products

Product records contain category and physical attributes.

In [12]:
display(products.head())

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


### Sellers

Seller records contain seller identifiers and location fields.

In [13]:
display(sellers.head())

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


### Geolocation

Geolocation records contain postal-code prefixes and geographic coordinates.

In [14]:
display(geolocation.head())

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


### Category Translation

Translation connects the original product category name with its English name.

In [15]:
display(translation.head())

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


# 10. Column Overview

In [16]:
column_overview = pd.DataFrame([
    {"Table": name, "Column": column}
    for name, table in tables.items()
    for column in table.columns
])
display(column_overview)

,Table,Column
0,customers,customer_id
1,customers,customer_unique_id
2,customers,customer_zip_code_prefix
3,customers,customer_city
4,customers,customer_state
5,orders,order_id
6,orders,customer_id
7,orders,order_status
8,orders,order_purchase_timestamp
9,orders,order_approved_at


# 11. Data Types

In [17]:
data_types = pd.DataFrame([
    {"Table": name, "Column": column, "Data Type": str(table[column].dtype)}
    for name, table in tables.items()
    for column in table.columns
])
display(data_types)

,Table,Column,Data Type
0,customers,customer_id,str
1,customers,customer_unique_id,str
2,customers,customer_zip_code_prefix,int64
3,customers,customer_city,str
4,customers,customer_state,str
5,orders,order_id,str
6,orders,customer_id,str
7,orders,order_status,str
8,orders,order_purchase_timestamp,str
9,orders,order_approved_at,str


### Data Type Interpretation

Several date fields are initially loaded as text. They will be converted to datetime when date-based analysis begins.

No type corrections are applied to the raw tables in this notebook.

# 12. Data Dictionary

In [18]:
data_dictionary_rows = [
    ["customers", "customer_id", "Identifier for a customer record.", "Connects customer records to orders."],
    ["customers", "customer_unique_id", "Identifier for the unique customer.", "Supports customer-level analysis."],
    ["customers", "customer_zip_code_prefix", "Customer postal-code prefix.", "Connects customer location to geolocation data."],
    ["customers", "customer_city", "Customer city.", "Describes customer location."],
    ["customers", "customer_state", "Customer state.", "Describes customer location."],

    ["orders", "order_id", "Identifier for an order.", "Connects order-level records across tables."],
    ["orders", "customer_id", "Customer identifier linked to the order.", "Connects orders to customers."],
    ["orders", "order_status", "Stored status of the order.", "Describes order state."],
    ["orders", "order_purchase_timestamp", "Order purchase timestamp.", "Marks the purchase event."],
    ["orders", "order_approved_at", "Order approval timestamp.", "Marks approval."],
    ["orders", "order_delivered_carrier_date", "Date the order was handed to the carrier.", "Describes carrier handoff."],
    ["orders", "order_delivered_customer_date", "Date the order was delivered to the customer.", "Describes customer delivery."],
    ["orders", "order_estimated_delivery_date", "Estimated delivery date.", "Represents the planned delivery date."],

    ["order_items", "order_id", "Order identifier.", "Connects items to orders."],
    ["order_items", "order_item_id", "Item sequence within an order.", "Identifies an item position within an order."],
    ["order_items", "product_id", "Product identifier.", "Connects items to products."],
    ["order_items", "seller_id", "Seller identifier.", "Connects items to sellers."],
    ["order_items", "shipping_limit_date", "Shipping limit timestamp.", "Provides an item-level shipping deadline."],
    ["order_items", "price", "Item price.", "Represents the item price."],
    ["order_items", "freight_value", "Freight value for the item.", "Represents item-level freight cost."],

    ["payments", "order_id", "Order identifier.", "Connects payments to orders."],
    ["payments", "payment_sequential", "Payment sequence number.", "Identifies payment order within an order."],
    ["payments", "payment_type", "Payment method.", "Describes how the payment was made."],
    ["payments", "payment_installments", "Number of installments.", "Describes payment installment structure."],
    ["payments", "payment_value", "Payment value.", "Represents the recorded payment amount."],

    ["reviews", "review_id", "Review identifier.", "Identifies the review record."],
    ["reviews", "order_id", "Order identifier.", "Connects reviews to orders."],
    ["reviews", "review_score", "Review score.", "Represents the customer review rating."],
    ["reviews", "review_comment_title", "Review comment title.", "Stores optional review text."],
    ["reviews", "review_comment_message", "Review comment message.", "Stores optional review text."],
    ["reviews", "review_creation_date", "Review creation date.", "Dates the review record."],
    ["reviews", "review_answer_timestamp", "Review answer timestamp.", "Dates the response to the review."],

    ["products", "product_id", "Product identifier.", "Connects products to order items."],
    ["products", "product_category_name", "Original product category name.", "Describes product category."],
    ["products", "product_name_lenght", "Length of the product name.", "Describes product-name size."],
    ["products", "product_description_lenght", "Length of the product description.", "Describes product-description size."],
    ["products", "product_photos_qty", "Number of product photos.", "Describes product listing content."],
    ["products", "product_weight_g", "Product weight in grams.", "Describes physical product attributes."],
    ["products", "product_length_cm", "Product length in centimeters.", "Describes physical product attributes."],
    ["products", "product_height_cm", "Product height in centimeters.", "Describes physical product attributes."],
    ["products", "product_width_cm", "Product width in centimeters.", "Describes physical product attributes."],

    ["sellers", "seller_id", "Seller identifier.", "Connects sellers to order items."],
    ["sellers", "seller_zip_code_prefix", "Seller postal-code prefix.", "Connects seller location to geolocation data."],
    ["sellers", "seller_city", "Seller city.", "Describes seller location."],
    ["sellers", "seller_state", "Seller state.", "Describes seller location."],

    ["geolocation", "geolocation_zip_code_prefix", "Postal-code prefix.", "Connects geographic records to customer and seller locations."],
    ["geolocation", "geolocation_lat", "Latitude.", "Geographic coordinate."],
    ["geolocation", "geolocation_lng", "Longitude.", "Geographic coordinate."],
    ["geolocation", "geolocation_city", "Geolocation city.", "Geographic reference."],
    ["geolocation", "geolocation_state", "Geolocation state.", "Geographic reference."],

    ["translation", "product_category_name", "Original category name.", "Matches the product category."],
    ["translation", "product_category_name_english", "English category name.", "Provides an English category label."],
]

data_dictionary = pd.DataFrame(
    data_dictionary_rows,
    columns=["Table", "Column", "Description", "Business Role"]
)
display(data_dictionary)

,Table,Column,Description,Business Role
0,customers,customer_id,Identifier for a customer record.,Connects customer records to orders.
1,customers,customer_unique_id,Identifier for the unique customer.,Supports customer-level analysis.
2,customers,customer_zip_code_prefix,Customer postal-code prefix.,Connects customer location to geolocation data.
3,customers,customer_city,Customer city.,Describes customer location.
4,customers,customer_state,Customer state.,Describes customer location.
5,orders,order_id,Identifier for an order.,Connects order-level records across tables.
6,orders,customer_id,Customer identifier linked to the order.,Connects orders to customers.
7,orders,order_status,Stored status of the order.,Describes order state.
8,orders,order_purchase_timestamp,Order purchase timestamp.,Marks the purchase event.
9,orders,order_approved_at,Order approval timestamp.,Marks approval.


# 13. Table Relationships

```text
Customers
    |
    | customer_id
    v
Orders
    |
    +------------ Payments
    |
    +------------ Reviews
    |
    +------------ Order Items
                      |
                      +------------ Products
                      |
                      +------------ Sellers

Products
    |
    | product_category_name
    v
Category Translation

Customers / Sellers
    |
    | postal-code prefix
    v
Geolocation
```

In [19]:
relationship_columns = pd.DataFrame([
    ["orders.customer_id -> customers.customer_id",
     "customer_id" in orders.columns and "customer_id" in customers.columns],
    ["order_items.order_id -> orders.order_id",
     "order_id" in order_items.columns and "order_id" in orders.columns],
    ["order_items.product_id -> products.product_id",
     "product_id" in order_items.columns and "product_id" in products.columns],
    ["order_items.seller_id -> sellers.seller_id",
     "seller_id" in order_items.columns and "seller_id" in sellers.columns],
    ["payments.order_id -> orders.order_id",
     "order_id" in payments.columns and "order_id" in orders.columns],
    ["reviews.order_id -> orders.order_id",
     "order_id" in reviews.columns and "order_id" in orders.columns],
    ["products.product_category_name -> translation.product_category_name",
     "product_category_name" in products.columns and "product_category_name" in translation.columns],
    ["customers.customer_zip_code_prefix -> geolocation.geolocation_zip_code_prefix",
     "customer_zip_code_prefix" in customers.columns and "geolocation_zip_code_prefix" in geolocation.columns],
    ["sellers.seller_zip_code_prefix -> geolocation.geolocation_zip_code_prefix",
     "seller_zip_code_prefix" in sellers.columns and "geolocation_zip_code_prefix" in geolocation.columns],
], columns=["Relationship", "Required Columns Found"])
display(relationship_columns)

,Relationship,Required Columns Found
0,orders.customer_id -> customers.customer_id,True
1,order_items.order_id -> orders.order_id,True
2,order_items.product_id -> products.product_id,True
3,order_items.seller_id -> sellers.seller_id,True
4,payments.order_id -> orders.order_id,True
5,reviews.order_id -> orders.order_id,True
6,products.product_category_name -> translation....,True
7,customers.customer_zip_code_prefix -> geolocat...,True
8,sellers.seller_zip_code_prefix -> geolocation....,True


# 14. Primary Key and Relationship Checks

In [20]:
important_keys = [
    ("customers", "customer_id"),
    ("orders", "order_id"),
    ("products", "product_id"),
    ("sellers", "seller_id"),
]

key_rows = []
for table_name, key in important_keys:
    table = tables[table_name]
    key_rows.append({
        "Table": table_name,
        "Key": key,
        "Rows": len(table),
        "Unique Keys": table[key].nunique(dropna=True),
        "Duplicate Keys": table[key].duplicated().sum(),
    })

key_summary = pd.DataFrame(key_rows)
display(key_summary)

,Table,Key,Rows,Unique Keys,Duplicate Keys
0,customers,customer_id,99441,99441,0
1,orders,order_id,99441,99441,0
2,products,product_id,32951,32951,0
3,sellers,seller_id,3095,3095,0


In [21]:
relationship_map = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("order_items", "seller_id", "sellers", "seller_id"),
    ("payments", "order_id", "orders", "order_id"),
    ("reviews", "order_id", "orders", "order_id"),
    ("products", "product_category_name", "translation", "product_category_name"),
    ("customers", "customer_zip_code_prefix", "geolocation", "geolocation_zip_code_prefix"),
    ("sellers", "seller_zip_code_prefix", "geolocation", "geolocation_zip_code_prefix"),
]

relationship_rows = []
for child_table, child_col, parent_table, parent_col in relationship_map:
    child_values = set(tables[child_table][child_col].dropna().unique())
    parent_values = set(tables[parent_table][parent_col].dropna().unique())
    missing_parent_values = child_values - parent_values
    relationship_rows.append({
        "Relationship": f"{child_table}.{child_col} -> {parent_table}.{parent_col}",
        "Distinct Child Values": len(child_values),
        "Missing Parent Values": len(missing_parent_values),
    })

relationship_summary = pd.DataFrame(relationship_rows)
display(relationship_summary)

,Relationship,Distinct Child Values,Missing Parent Values
0,orders.customer_id -> customers.customer_id,99441,0
1,order_items.order_id -> orders.order_id,98666,0
2,order_items.product_id -> products.product_id,32951,0
3,order_items.seller_id -> sellers.seller_id,3095,0
4,payments.order_id -> orders.order_id,99440,0
5,reviews.order_id -> orders.order_id,98673,0
6,products.product_category_name -> translation....,73,2
7,customers.customer_zip_code_prefix -> geolocat...,14994,157
8,sellers.seller_zip_code_prefix -> geolocation....,2246,7


### Key and Relationship Interpretation

The key summary checks uniqueness of important identifiers.

The relationship summary checks whether distinct values in child tables can be found in their expected parent tables.

Any non-zero result is a candidate for investigation during Data Quality. It is not treated as a business conclusion.

# 15. Missing Values

In [22]:
missing_rows = []

for table_name, table in tables.items():
    for column in table.columns:
        missing_count = table[column].isna().sum()
        if missing_count > 0:
            missing_rows.append({
                "Table": table_name,
                "Column": column,
                "Missing Count": int(missing_count),
                "Missing Percentage": round(missing_count / len(table) * 100, 2),
            })

missing_summary = pd.DataFrame(missing_rows)

if not missing_summary.empty:
    missing_summary = missing_summary.sort_values(
        ["Missing Percentage", "Missing Count"],
        ascending=False
    ).reset_index(drop=True)

display(missing_summary)

,Table,Column,Missing Count,Missing Percentage
0,reviews,review_comment_title,87656,88.34
1,reviews,review_comment_message,58247,58.70
2,orders,order_delivered_customer_date,2965,2.98
3,products,product_category_name,610,1.85
4,products,product_name_lenght,610,1.85
5,products,product_description_lenght,610,1.85
6,products,product_photos_qty,610,1.85
7,orders,order_delivered_carrier_date,1783,1.79
8,orders,order_approved_at,160,0.16
9,products,product_weight_g,2,0.01


### Missing Values Interpretation

Missing values do not automatically mean that a record is invalid.

Some missing fields may represent information that was not recorded or an event that did not occur.

At this stage, missing values are identified and measured only. Their causes and impact will be investigated during Data Quality.

# 16. Duplicate Records

In [23]:
duplicate_summary = pd.DataFrame([
    {
        "Table": table_name,
        "Rows": len(table),
        "Exact Duplicate Rows": int(table.duplicated().sum()),
    }
    for table_name, table in tables.items()
])

display(duplicate_summary)

,Table,Rows,Exact Duplicate Rows
0,customers,99441,0
1,orders,99441,0
2,order_items,112650,0
3,payments,103886,0
4,reviews,99224,0
5,products,32951,0
6,sellers,3095,0
7,geolocation,1000163,261831
8,translation,71,0


### Duplicate Records Interpretation

An exact duplicate row is different from a repeated identifier.

For example, an `order_id` can legitimately appear multiple times in Order Items because one order can contain multiple items.

No records are removed at this stage.

# 17. Order Status

In [24]:
order_status_summary = (
    orders["order_status"]
    .value_counts(dropna=False)
    .rename_axis("Order Status")
    .reset_index(name="Orders")
)

display(order_status_summary)

,Order Status,Orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


### Order Status Interpretation

The table shows the order statuses stored in the dataset and their counts.

At this stage, the values are treated as structural information only.

# 18. Order Date Range

In [25]:
lifecycle_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for column in lifecycle_columns:
    orders[column] = pd.to_datetime(orders[column], errors="coerce")

date_summary = pd.DataFrame([
    ["First purchase date", orders["order_purchase_timestamp"].min()],
    ["Last purchase date", orders["order_purchase_timestamp"].max()],
], columns=["Metric", "Value"])

orders_by_year = (
    orders["order_purchase_timestamp"]
    .dt.year
    .value_counts()
    .sort_index()
    .rename_axis("Year")
    .reset_index(name="Orders")
)

display(date_summary)
display(orders_by_year)

,Metric,Value
0,First purchase date,2016-09-04 21:15:19
1,Last purchase date,2018-10-17 17:30:18


,Year,Orders
0,2016,329
1,2017,45101
2,2018,54011


### Date Range Interpretation

The purchase timestamp defines the main time period represented by the dataset.

The other timestamps describe different stages of the order lifecycle.

These dates will support later time-based analysis.

# 19. Order Lifecycle

### Actual Order Lifecycle

```text
Purchase
   ↓
Approval
   ↓
Carrier Handoff
   ↓
Customer Delivery
```

`order_estimated_delivery_date` is an expected delivery date, not an actual lifecycle event.

No delivery-performance KPI is calculated in this notebook.

# 20. Geographic Structure

In [26]:
geographic_summary = pd.DataFrame([
    ["Customer states", customers["customer_state"].nunique()],
    ["Customer cities", customers["customer_city"].nunique()],
    ["Customer postal-code prefixes", customers["customer_zip_code_prefix"].nunique()],
    ["Seller states", sellers["seller_state"].nunique()],
    ["Seller cities", sellers["seller_city"].nunique()],
    ["Seller postal-code prefixes", sellers["seller_zip_code_prefix"].nunique()],
    ["Geolocation postal-code prefixes", geolocation["geolocation_zip_code_prefix"].nunique()],
    ["Geolocation cities", geolocation["geolocation_city"].nunique()],
], columns=["Metric", "Value"])

display(geographic_summary)

,Metric,Value
0,Customer states,27
1,Customer cities,4119
2,Customer postal-code prefixes,14994
3,Seller states,23
4,Seller cities,611
5,Seller postal-code prefixes,2246
6,Geolocation postal-code prefixes,19015
7,Geolocation cities,8011


### Geographic Structure Interpretation

Customer and seller tables contain location attributes, while the geolocation table provides additional geographic information.

This structure can support geographic analysis later.

# 21. Category Translation Structure

In [27]:
product_categories = set(products["product_category_name"].dropna())
translation_categories = set(translation["product_category_name"].dropna())

category_summary = pd.DataFrame([
    ["Product categories", len(product_categories)],
    ["Translation categories", len(translation_categories)],
    ["Product categories without translation", len(product_categories - translation_categories)],
], columns=["Metric", "Value"])

missing_translation = pd.DataFrame({
    "Product Category Without Translation": sorted(product_categories - translation_categories)
})

display(category_summary)
display(missing_translation.head(20))

,Metric,Value
0,Product categories,73
1,Translation categories,71
2,Product categories without translation,2


,Product Category Without Translation
0,pc_gamer
1,portateis_cozinha_e_preparadores_de_alimentos


### Category Translation Interpretation

The comparison shows how product categories in the Products table connect to the translation table.

Unmatched categories are candidates for later investigation if English category labels are required.

# 22. Initial Structural Observations

- The dataset contains nine related tables.
- Orders are the central order-level table.
- Customers connect to orders through `customer_id`.
- Orders can contain multiple order items.
- Order items connect products and sellers to orders.
- Payments and reviews connect to orders.
- Product categories can connect to English translations.
- Customer and seller postal-code prefixes can connect to geolocation information.
- Multiple timestamps describe different stages of the order lifecycle.
- Missing values and exact duplicate rows were identified without modifying the raw data.

These are structural observations only. Business findings will be developed during EDA.

# 23. What We Learned

1. The main tables and their roles are now identified.
2. The data grain of each table is documented.
3. The main identifiers and relationships are understood.
4. Orders connect customers, items, payments, and reviews.
5. Order items connect products and sellers.
6. Geographic information is represented through customer, seller, and geolocation tables.
7. Product categories can be connected to English translations.
8. Order lifecycle timestamps are available for later time-based analysis.
9. Missing values and duplicate records have been measured without changing the raw data.

# 24. Dataset Understanding Summary

In [28]:
final_overview = pd.DataFrame([
    ["Number of Tables", len(tables)],
    ["Total Orders", len(orders)],
    ["Total Customers", len(customers)],
    ["Total Order Items", len(order_items)],
    ["Total Products", len(products)],
    ["Total Sellers", len(sellers)],
    ["Total Payments", len(payments)],
    ["Total Reviews", len(reviews)],
], columns=["Metric", "Value"])

display(final_overview)

,Metric,Value
0,Number of Tables,9
1,Total Orders,99441
2,Total Customers,99441
3,Total Order Items,112650
4,Total Products,32951
5,Total Sellers,3095
6,Total Payments,103886
7,Total Reviews,99224


The Dataset Understanding stage is complete.

The raw data has been inspected structurally without deleting, filling, or transforming records.

The next step is Data Quality.

# 25. Next Stage: Data Quality

The next notebook will investigate:

1. Missing values and their possible causes.
2. Duplicate records and repeated identifiers.
3. Invalid or inconsistent values.
4. Date and timestamp consistency.
5. Relationship integrity.
6. Numerical ranges and possible anomalies.

Only after Data Quality will we move to Exploratory Data Analysis and business questions.